## 1)chargement du dataset

In [1]:
from datasets import load_dataset

ds = load_dataset("ai4privacy/open-pii-masking-500k-ai4privacy")
print(ds)

README.md: 0.00B [00:00, ?B/s]

data/train/train.jsonl:   0%|          | 0.00/566M [00:00<?, ?B/s]

data/validation/test.jsonl:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/464150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/116077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 464150
    })
    validation: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 116077
    })
})


## 2) garder uniquement le français

In [2]:
ds_fr = ds.filter(lambda ex: ex["language"] == "fr")

print(ds_fr)
print("Train FR :", len(ds_fr["train"]))
print("Validation FR :", len(ds_fr["validation"]))

Filter:   0%|          | 0/464150 [00:00<?, ? examples/s]

Filter:   0%|          | 0/116077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 89670
    })
    validation: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 22466
    })
})
Train FR : 89670
Validation FR : 22466


## 3) Exemple du dataset

In [3]:
for i in range(3):
    ex = ds_fr["train"][i]
    print("ORIGINAL :", ex["source_text"])
    print("MASQUÉ   :", ex["masked_text"])
    print("ENTITÉS  :", [(e["label"], e["value"]) for e in ex["privacy_mask"]])
    print("-" * 80)

ORIGINAL : Le sexe de Oswald est Masculin.
MASQUÉ   : Le sexe de [GIVENNAME_1] est [SEX_1].
ENTITÉS  : [('GIVENNAME', 'Oswald'), ('SEX', 'Masculin')]
--------------------------------------------------------------------------------
ORIGINAL : Le 30 minimum pour participer au projet est de 63 ans.
MASQUÉ   : Le [AGE_2] minimum pour participer au projet est de [AGE_1] ans.
ENTITÉS  : [('AGE', '30'), ('AGE', '63')]
--------------------------------------------------------------------------------
ORIGINAL : La L9Y de Meyriam Zeller-Bieri est nécessaire pour l'envoi de la correspondance relative à la demande de 41NYJ7SHEJ.
MASQUÉ   : La [ZIPCODE_1] de [GIVENNAME_1] [SURNAME_1] est nécessaire pour l'envoi de la correspondance relative à la demande de [IDCARDNUM_1].
ENTITÉS  : [('ZIPCODE', 'L9Y'), ('GIVENNAME', 'Meyriam'), ('SURNAME', 'Zeller-Bieri'), ('IDCARDNUM', '41NYJ7SHEJ')]
--------------------------------------------------------------------------------


Inspection manuelle : données synthétiques avec des artefacts de génération (valeurs à la place de noms communs, codes postaux non français, numérotation incohérente des étiquettes). Décisions : filtre qualité, normalisation des étiquettes, jeu de test réaliste écrit à la main.

In [4]:
ds_fr["train"][0]

{'source_text': 'Le sexe de Oswald est Masculin.',
 'masked_text': 'Le sexe de [GIVENNAME_1] est [SEX_1].',
 'privacy_mask': [{'label': 'GIVENNAME',
   'start': 11,
   'end': 17,
   'value': 'Oswald',
   'label_index': 1},
  {'label': 'SEX',
   'start': 22,
   'end': 30,
   'value': 'Masculin',
   'label_index': 1}],
 'split': 'train',
 'uid': 5403672,
 'language': 'fr',
 'region': 'FR',
 'script': 'Latn',
 'mbert_tokens': ['Le',
  'sexe',
  'de',
  'Oswald',
  'est',
  'Mas',
  '##cul',
  '##in',
  '.'],
 'mbert_token_classes': ['O',
  'O',
  'O',
  'B-GIVENNAME',
  'O',
  'B-SEX',
  'I-SEX',
  'I-SEX',
  'O']}

## 4)quels types de données personnelles ?

In [5]:
from collections import Counter

labels = Counter(
    e["label"] for ex in ds_fr["train"] for e in ex["privacy_mask"]
)
for label, n in labels.most_common():
    print(f"{label:<25} {n}")

GIVENNAME                 65257
SURNAME                   24607
TELEPHONENUM              14982
CITY                      13776
TIME                      12657
EMAIL                     10957
DATE                      9697
STREET                    9110
BUILDINGNUM               8118
IDCARDNUM                 7264
TITLE                     6570
AGE                       4340
ZIPCODE                   3552
PASSPORTNUM               3534
SEX                       2126
TAXNUM                    2117
SOCIALNUM                 1945
CREDITCARDNUMBER          1910
DRIVERLICENSENUM          1796
GENDER                    1432


Distribution très déséquilibrée : les prénoms et noms représentent ~44 % des entités, et les identifiants sensibles (carte bancaire, sécurité sociale) moins de 1 % chacun. Décisions : échantillonnage qui garantit la présence des types rares, évaluation par type d'entité, fusion éventuelle de SEX/GENDER.

## 5)longueur des textes

In [6]:
import pandas as pd

longueurs = pd.Series([len(t) for t in ds_fr["train"]["source_text"]])
print(longueurs.describe())

nb_entites = pd.Series([len(m) for m in ds_fr["train"]["privacy_mask"]])
print(nb_entites.describe())

count    89670.000000
mean       126.207360
std         51.199407
min          0.000000
25%         91.000000
50%        120.000000
75%        156.000000
max       1170.000000
dtype: float64
count    89670.000000
mean         2.294491
std          1.304025
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         23.000000
dtype: float64


## 6) doublons

In [7]:
textes = ds_fr["train"]["source_text"]
print("Textes :", len(textes))
print("Textes uniques :", len(set(textes)))
print("Doublons exacts :", len(textes) - len(set(textes)))

# Fuite train -> validation : textes présents dans les deux
fuite = set(textes) & set(ds_fr["validation"]["source_text"])
print("Textes à la fois en train ET en validation :", len(fuite))

Textes : 89670
Textes uniques : 89490
Doublons exacts : 180
Textes à la fois en train ET en validation : 51


## 7)Quasi-doublons

In [8]:
import re

def gabarit(texte_masque):
    # [GIVENNAME_1] -> [GIVENNAME] : on retire les numéros
    return re.sub(r"\[([A-Z]+)_\d+\]", r"[\1]", texte_masque)

gabarits = [gabarit(t) for t in ds_fr["train"]["masked_text"]]
compte = Counter(gabarits)

print("Textes :", len(gabarits))
print("Gabarits distincts :", len(compte))
print("\nLes 5 gabarits les plus répétés :")
for g, n in compte.most_common(5):
    print(f"{n:>5} x  {g}")

Textes : 89670
Gabarits distincts : 87350

Les 5 gabarits les plus répétés :
   75 x  Le numéro de téléphone de [GIVENNAME] est [TELEPHONENUM].
   36 x  ...
   20 x  En raison de nouvelles réglementations, nous devons mettre à jour nos dossiers avec votre [TAXNUM] et une preuve de [AGE]. Veuillez soumettre ces documents avant votre prochaine séance.
   20 x  La date de naissance de [GIVENNAME] est [DATE].
   19 x  Mon numéro de téléphone est [TELEPHONENUM].


## 8) inspections des cas extrêmes

In [9]:
train = ds_fr["train"]

vides = [ex for ex in train if len(ex["source_text"].strip()) == 0]
print("Textes vides :", len(vides))

sans_entite = [ex for ex in train if len(ex["privacy_mask"]) == 0]
print("Textes sans entité :", len(sans_entite))
for ex in sans_entite[:3]:
    print("  →", ex["source_text"])

plus_charge = max(train, key=lambda ex: len(ex["privacy_mask"]))
print("\nTexte avec le plus d'entités :")
print(plus_charge["source_text"])

Textes vides : 2
Textes sans entité : 3100
  → Nous vous remercions de votre participation à notre atelier de design de jardins monastiques médiévaux ! Nous espérons que vous avez apprécié l'expérience et que vous vous êtes inspiré pour créer un jardin magnifique.
  → Je vous remercie de votre intérêt pour l'installation de la station de recharge solaire. Nous vous contacterons bientôt pour discuter des détails.
  → Reminder: il est important de pratiquer la mindfulness régulièrement. Voir les instructions jointes pour les détails.

Texte avec le plus d'entités :
06:05:32 Tsvetomir: 'Hello Sugirthan, je voudrais discuter de mon projet de bijoux en LED.';
11:16 PM Telly Ghebreamlakh Galletto: 'Super, j'adore les bijoux lumineux !'
4h04 0152-46-149.5773: 'Moi aussi, j'ai même un exemple de projet que j'ai réalisé récemment.';
11h27 Yuyan Preprotic: 'Wow, c'est magnifique ! Je serais ravi de voir la manière dont vous avez procédé.';
4h35 Lunna Napat Krol: 'Bien sûr, je vous envoie un e-ma

## Bilan de l'exploration

Le jeu de données `ai4privacy/open-pii-masking-500k-ai4privacy` (licence CC-BY-4.0) contient 464 150 exemples d'entraînement et 116 077 de validation, répartis sur 8 langues. Après filtrage sur le français, il reste 89 670 exemples en train et 22 466 en validation, soit environ 19 % du total. Chaque exemple associe un texte original (`source_text`), sa version anonymisée (`masked_text`) et la liste des données personnelles repérées, avec leur type et leur position (`privacy_mask`). Ce volume dépasse largement les besoins d'un fine-tuning LoRA sur un petit modèle : un sous-ensemble de quelques milliers d'exemples soigneusement sélectionnés sera retenu.

L'inspection manuelle révèle que les données sont synthétiques et comportent des artefacts de génération : certaines valeurs ont remplacé des noms communs et rendent la phrase absurde (« Le 30 minimum pour participer… », « une preuve de [AGE] »), certains codes postaux ne sont pas au format français, et la numérotation des étiquettes (`[AGE_2]` avant `[AGE_1]`) n'est pas cohérente. Les numéros seront retirés (`[GIVENNAME_1]` → `[GIVENNAME]`), et un filtre de qualité sera conçu pour écarter les phrases incohérentes.

Le texte moyen est court (médiane de 120 caractères, 75 % sous 156, maximum 1 170) et contient en moyenne 2,3 données personnelles, ce qui garantit des séquences courtes et un entraînement rapide ; une longueur maximale en tokens sera fixée pour écarter les rares textes très longs. La distribution des types d'entités est très déséquilibrée : prénoms et noms représentent environ 44 % des entités, tandis que les identifiants les plus sensibles (carte bancaire, numéro de sécurité sociale, permis de conduire) pèsent chacun moins de 1 %. L'échantillonnage devra donc garantir la présence des types rares, et l'évaluation sera menée type par type. Les étiquettes SEX et GENDER, quasi identiques, seront fusionnées.

Côté qualité, 2 textes sont vides et 36 exemples ont une version masquée réduite à « ... » : ils seront supprimés. On compte 180 doublons exacts dans le train (0,2 %) et 51 textes présents à la fois en train et en validation ; ces derniers seront retirés du train pour éviter toute fuite qui fausserait l'évaluation. L'hypothèse de nombreux quasi-doublons, liée à la génération synthétique, ne se vérifie pas : en remplaçant les données personnelles par leur type, on obtient 87 350 structures de phrases distinctes pour 89 670 textes, soit seulement 2,6 % de répétitions. Le nombre d'exemples par structure sera plafonné à 3. Enfin, 3 100 textes (3,5 %) ne contiennent aucune donnée personnelle ; environ 5 % de ce type d'exemples seront conservés pour apprendre au modèle à ne rien masquer lorsqu'il n'y a rien à masquer.

Deux limites sont à garder en tête pour la suite : les noms présents dans le jeu de données sont très internationaux et peu représentatifs des noms français courants, et les heures (TIME) sont considérées comme des données personnelles, ce qui reste discutable. Un petit jeu de test composé de textes français réalistes, écrits à la main, permettra de vérifier le comportement du modèle hors de la distribution synthétique.